# 6. Is it fair to developing countries? And an honest uncertainty band

**Life Expectancy Predictor · notebook 6 of 7**

Notebook 5 showed that the hardest countries are almost all developing countries. That raises a fairness question: is the model *worse* for developing countries, or are they simply harder cases? This notebook answers it carefully, then turns to a related promise the model makes: an uncertainty band around each prediction. If the band is too narrow for one group, that is a fairness problem of its own, and it turns out to be the one that needed fixing.

Note: the developed/developing label is **not** one of the model's inputs. The model has never seen it. We only use it here to grade the results.

## Step 1: Fix the labels first

**What we are looking for.** Notebook 1 found six advanced economies tagged "Developing" in the source file. Before comparing groups, they are moved to "Developed", and we check how much that changes the comparison.

In [1]:
# Setup: find the project folder and load the project's own code and settings.
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config_loader import load_config

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
config = load_config(str(ROOT / "config" / "who_life_expectancy.yaml"), base_dir=str(ROOT))

def results(name):
    """Load one of the saved results files in model/."""
    return json.loads((ROOT / "model" / f"{name}.json").read_text(encoding="utf-8"))

print("Project folder:", ROOT.name)
oof = pd.read_csv("oof_predictions.csv")            # each country predicted while unseen (notebook 3)
oof["miss"] = (oof["oof_prediction"] - oof["truth"]).abs()
status = pd.read_csv(ROOT / "data" / "clean_data.csv")[["Country", "Year", "Status"]]
oof = oof.merge(status, on=["Country", "Year"])

fair = results("fairness_check")
relabel = fair["relabelled_countries"]
oof["Status_fixed"] = np.where(oof["Country"].isin(relabel), "Developed", oof["Status"])

def gap(col):
    g = oof.groupby(col)["miss"].mean()
    return g["Developed"], g["Developing"], g["Developing"] - g["Developed"]
rows = {"Labels as in the file": gap("Status"), "Six countries relabelled": gap("Status_fixed")}
pd.DataFrame(rows, index=["Developed: average miss", "Developing: average miss", "Gap (years)"]).T.round(2)

Project folder: nb


,Developed: average miss,Developing: average miss,Gap (years)
Labels as in the file,1.70,2.23,0.54
Six countries relabelled,1.81,2.22,0.41


**How to read it.** Average miss for each group, before and after moving Canada, Finland, France, Greece, Israel and Korea to "Developed", and the gap between the groups.

**What it means.** The gap shrinks from about 0.54 to 0.41 years when the labels are corrected. A quarter of the apparent unfairness was a labelling error in the source data, not the model. The rest of this notebook uses the corrected labels.

**What's next and why.** A 0.41-year gap could still be chance. Check.

## Step 2: Is the gap real, and is it the model's fault?

**What we are looking for.** Two things. First, whether a 0.41-year gap is more than noise: redraw the set of countries many times (bootstrap) and see whether the gap stays above zero. Second, whether the gap is caused by developing countries simply being a harder, more varied group: compare the two groups only among countries with *similar* life expectancy.

In [2]:
rng = np.random.default_rng(0)
per_country = oof.groupby(["Country", "Status_fixed"])["miss"].mean().reset_index()
dev = per_country[per_country.Status_fixed == "Developed"]["miss"].to_numpy()
dvg = per_country[per_country.Status_fixed == "Developing"]["miss"].to_numpy()
gaps = np.array([rng.choice(dvg, len(dvg)).mean() - rng.choice(dev, len(dev)).mean() for _ in range(2000)])
lo, hi = np.percentile(gaps, [2.5, 97.5])
print(f"Gap between groups: {dvg.mean() - dev.mean():.2f} years; in 95% of redraws it falls between {lo:.2f} and {hi:.2f}; "
      f"above zero in {(gaps > 0).mean():.1%} of redraws.\n")

oof["band"] = pd.cut(oof["truth"], [0, 60, 65, 70, 75, 80, 100], labels=["under 60", "60-65", "65-70", "70-75", "75-80", "80+"])
by_band = oof.groupby(["band", "Status_fixed"], observed=True)["miss"].agg(["mean", "count"]).unstack()
by_band.columns = [f"{a} ({b})" for b, a in by_band.columns]
by_band = by_band.rename(columns={"Developed (mean)": "Developed: miss", "Developing (mean)": "Developing: miss",
                                  "Developed (count)": "Developed: rows", "Developing (count)": "Developing: rows"})
by_band.round(2)

Gap between groups: 0.41 years; in 95% of redraws it falls between 0.08 and 0.73; above zero in 99.2% of redraws.



,Developed: miss,Developing: miss,Developed: rows,Developing: rows
band,,,,
under 60,NaN,2.98,NaN,594.0
60-65,NaN,2.19,NaN,270.0
65-70,1.91,2.34,1.0,443.0
70-75,1.21,1.51,89.0,707.0
75-80,1.65,2.25,268.0,303.0
80+,2.21,3.44,250.0,3.0


**How to read it.** The first line is the bootstrap: the gap is real, not a fluke of which countries happened to be in the data (it stays above zero in essentially every redraw). The table compares the groups *within bands of similar life expectancy*, which removes the "developing countries are just a harder group" explanation.

**What it means.** Even among countries living 75-80 years, developing countries are missed by about 0.6 years more than developed ones, and the same holds in the 70-75 band. (The bands below 65 have no developed countries, and the 80+ band has only three developing rows, so those cannot be compared.) The gap is modest (half a year against an average miss of two) but it is consistent, and it is not explained by the groups being different in difficulty. A likely reason is reporting quality: statistics from developing countries are, on average, noisier, and the model cannot know which numbers to trust. This is recorded as a known limitation rather than hidden.

**What's next and why.** Accuracy is one kind of fairness. The other is honesty about uncertainty.

## Step 3: The uncertainty band, and who it short-changes

**What we are looking for.** Alongside each prediction, the model reports a range: "probably between 63 and 70". It comes from the spread of the 200 trees: if they disagree, the range is wide. The project states that the band should contain the true value **80% of the time**. Here we measure whether it does, for each group, with every country unseen.

In [3]:
from data_prep import get_unclipped_df
from train_model import rf_pipeline
from sklearn.model_selection import GroupKFold

df = get_unclipped_df(config)
X = df[config.predictor_columns].to_numpy(); y = df[config.target_column].to_numpy(); groups = df["Country"].to_numpy()
mp = config.model_params
lo_b, hi_b = np.zeros(len(y)), np.zeros(len(y))
for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    m = rf_pipeline(config, n_estimators=mp["n_estimators"], random_state=mp["random_state"], n_jobs=-1).fit(X[tr], y[tr])
    X_te_clipped = m.named_steps["clip"].transform(X[te])
    trees = np.stack([t.predict(X_te_clipped) for t in m.named_steps["rf"].estimators_])   # 200 opinions per row
    lo_b[te], hi_b[te] = np.percentile(trees, 10, axis=0), np.percentile(trees, 90, axis=0)

band = pd.DataFrame({"Country": groups, "Year": df["Year"].values, "truth": y, "lo": lo_b, "hi": hi_b})
band = band.merge(oof[["Country", "Year", "Status_fixed"]], on=["Country", "Year"])
band["inside"] = (band.truth >= band.lo) & (band.truth <= band.hi)
band["width"] = band.hi - band.lo
summary = band.groupby("Status_fixed").agg(**{"Truth inside the band (%)": ("inside", lambda s: 100 * s.mean()), "Band width (years)": ("width", "mean")})
summary.loc["All"] = [100 * band.inside.mean(), band.width.mean()]
summary.round(1)

,Truth inside the band (%),Band width (years)
Status_fixed,,
Developed,80.8,5.6
Developing,76.3,7.0
All,77.2,6.7


**How to read it.** For each group: how often the true value actually fell inside the stated 80% band, and how wide the band is on average.

**What it means.** For developed countries the band keeps its promise (about 81%). For developing countries it does not: the truth lands inside only about 76% of the time. The model is *overconfident* about developing countries: it gives a band that sounds like "80% sure" but is really "76% sure". That is a second, subtler kind of unfairness, and unlike the accuracy gap it can be fixed, because it is about how the band is *scaled*, not about which country is which.

**What's next and why.** Stretch the band until it is honest, separately for each group, and test the stretch fairly.

## Step 4: Stretching the band until it is honest

**What we are looking for.** A multiplier for each group's band width, chosen so that coverage hits 80%. To keep the test honest, the multipliers are learned on some countries and tested on others (cross-fitting), so the result below is what a new country would get. The project also tried one multiplier for everyone, to see whether a per-group one is really needed.

In [4]:
cal = results("interval_calibration")
ev = cal["evaluation"]
rows = {"As is": ev["A_raw"], "One stretch for everyone": ev["B_one_factor"], "A stretch per group": ev["C_per_group"]}
tbl = pd.DataFrame({k: [v["All"]["coverage_pct"], v["Developed"]["coverage_pct"], v["Developing"]["coverage_pct"],
                        v["Developed"]["width_years"], v["Developing"]["width_years"]] for k, v in rows.items()},
                   index=["Inside: all (%)", "Inside: developed (%)", "Inside: developing (%)", "Width: developed (yrs)", "Width: developing (yrs)"]).T
display(tbl)
f = cal["factors_fitted_on_all"]
print(f"Multipliers chosen on all data: developed ×{f['Developed']:.2f}, developing ×{f['Developing']:.3f}")

,Inside: all (%),Inside: developed (%),Inside: developing (%),Width: developed (yrs),Width: developing (yrs)
As is,77.2,80.8,76.3,5.55,6.98
One stretch for everyone,79.7,82.7,78.9,6.16,7.67
A stretch per group,80.1,81.1,79.9,5.81,7.83


Multipliers chosen on all data: developed ×1.00, developing ×1.127


**How to read it.** Three ways of scaling the band, with the share of true values inside it for each group, and the band's width.

**What it means.** One multiplier for everyone gets the overall number to 80% but does it by over-widening the developed band (83%) while the developing band is still short (79%): it fixes the average, not the fairness. A multiplier per group fixes both: developed 81%, developing 80%. The price is honesty, not accuracy: a developing country's band becomes about 13% wider (7.0 → 7.8 years), because that is how uncertain the model really is about it. The published model uses the per-group stretch.

**What's next and why.** Everything so far has been about the random forest. Notebook 7 asks the question an employer would: did you try anything else?

## Summary

| Question | Answer |
| --- | --- |
| Is the model less accurate for developing countries? | Yes, by about 0.4 years (after fixing six mislabelled countries), and the gap holds among countries with similar life expectancy |
| Is that chance? | No: it stays above zero in over 99% of bootstrap redraws |
| Can it be fixed? | Not from these inputs; likely a reporting-quality effect. Recorded as a limitation |
| Was the 80% band honest? | For developed countries yes (81%); for developing countries no (76%) |
| Fix | A per-group stretch of the band, tested cross-fitted: 81% / 80%. The developing band is 13% wider than before |